# Format Negotiation in Multi-Agent Systems

## The Problem

Imagine two AI agents built by different teams:
- **Agent A** (a reporting tool) produces data as XML
- **Agent B** (a dashboard) only understands JSON

If A just sends XML to B, B breaks. You could hardcode a conversion — but what if you add a third agent that speaks both? What if agent capabilities change over time?

**Format negotiation** solves this properly: before any data is sent, agents declare what they support and agree on a shared format at runtime. No hardcoding, no guessing.

## What you'll learn

- Why format negotiation matters in multi-agent systems
- How to implement a negotiate → convert → deliver protocol
- What happens when agents have no common format (safe fallback)
- How to observe all three scenarios: XML→JSON, JSON→XML, and no-overlap

## The 4-step protocol

```
Step 1: DECLARE   — each agent lists the formats it supports and its preference order
Step 2: NEGOTIATE — find the overlap; pick the best format based on sender preference
Step 3: CONVERT   — transform the payload if the agreed format differs from the source
Step 4: DELIVER   — send the converted payload, OR abort with FALLBACK if no overlap
```

---

### The pipeline

```
   Agent A                                    Agent B
(supported, pref)                          (supported, pref)
      │                                            │
      └─────────────────────┬──────────────────────┘
                             ▼
                    ┌────────────────┐
                    │  negotiate()   │   DECLARE + NEGOTIATE
                    └────────────────┘
                             │
              overlap found ◀┴▶ no overlap
                    │                   │
                    ▼                   ▼
           ┌────────────────┐    ┌────────────┐
           │ convert_bytes()│    │  FALLBACK  │
           └────────────────┘    └────────────┘
                    │  CONVERT
                    ▼
           ┌────────────────┐
           │ receiver.recv()│   DELIVER
           └────────────────┘
```

The sender never knows in advance which format will "win" — `negotiate()` decides that at runtime from both agents' declared support and preferences, and the whole thing short-circuits into `FALLBACK` the moment there's no common format.

## Step 1 — Imports

This demo uses only Python's standard library — no external packages needed. `json` and `xml.etree.ElementTree` are built in.

In [ ]:
import json
import xml.etree.ElementTree as ET
from typing import List, Dict, Any, Optional

print("Ready.")

## Step 2 — Format Converters

These are low-level helpers. Each one converts a Python `dict` to/from a byte string in the target format.

Why bytes? In a real system, agents communicate over a network. Payloads are byte streams — not Python objects. Keeping things as bytes makes this demo realistic.

| Function | What it does |
|---|---|
| `dict_to_json_bytes` | Serialises a dict to compact JSON bytes |
| `json_bytes_to_dict` | Parses JSON bytes back to a dict |
| `dict_to_xml_bytes` | Wraps each dict key as an XML element |
| `xml_bytes_to_dict` | Parses XML bytes back to a dict, preserving numeric types |

In [ ]:
def dict_to_json_bytes(d: Dict) -> bytes:
    return json.dumps(d, separators=(",", ":"), ensure_ascii=False).encode("utf-8")

def json_bytes_to_dict(b: bytes) -> Dict:
    return json.loads(b.decode("utf-8"))

def dict_to_xml_bytes(d: Dict) -> bytes:
    root = ET.Element("message")
    for k, v in d.items():
        child = ET.SubElement(root, str(k))
        child.text = str(v)
    return ET.tostring(root, encoding="utf-8")

def xml_bytes_to_dict(b: bytes) -> Dict:
    root = ET.fromstring(b.decode("utf-8"))
    out = {}
    for child in root:
        text = child.text or ""
        try:
            num = float(text)
            out[child.tag] = int(num) if num.is_integer() else num
        except ValueError:
            out[child.tag] = text
    return out

print("Converters defined.")

### Quick check — see the converters in action

Before moving on, let's verify the round-trip works correctly: `dict → XML bytes → dict` and `dict → JSON bytes → dict`.

In [ ]:
sample = {"id": 101, "title": "Quarterly Report", "amount": 123.45}

json_bytes = dict_to_json_bytes(sample)
xml_bytes  = dict_to_xml_bytes(sample)

print("JSON bytes:", json_bytes.decode())
print("XML bytes: ", xml_bytes.decode())
print()
print("JSON → dict:", json_bytes_to_dict(json_bytes))
print("XML  → dict:", xml_bytes_to_dict(xml_bytes))

Both round-trips should return the original dict with types preserved (`id` as int, `amount` as float).

---

## Step 3 — The Conversion Router

`convert_bytes` is the single entry point for all format transformations. It decodes the source payload to a dict, then re-encodes it in the destination format. If source and destination are the same, it's a no-op.

In [ ]:
def convert_bytes(src_fmt: str, dst_fmt: str, payload: bytes) -> bytes:
    src_fmt, dst_fmt = src_fmt.upper(), dst_fmt.upper()
    if src_fmt == dst_fmt:
        return payload  # nothing to do

    # Decode to intermediate dict
    if src_fmt == "JSON":
        d = json_bytes_to_dict(payload)
    elif src_fmt == "XML":
        d = xml_bytes_to_dict(payload)
    else:
        raise ValueError(f"Unsupported source format: {src_fmt}")

    # Re-encode in destination format
    if dst_fmt == "JSON":
        return dict_to_json_bytes(d)
    elif dst_fmt == "XML":
        return dict_to_xml_bytes(d)
    else:
        raise ValueError(f"Unsupported destination format: {dst_fmt}")

print("Converter router defined.")

## Step 4 — The Negotiation Algorithm

This is the brain of the system. Given what both sides support, it finds the best format to use.

### Algorithm
1. Compute the **intersection** of sender's and receiver's supported formats
2. If the intersection is empty → return `None` (triggers FALLBACK)
3. Otherwise, walk through the **sender's preference list** and pick the first format that's in the overlap
4. If none of the sender's preferences match, try the **receiver's preference list**

> **Why sender preference first?** The sender is doing the work of encoding. Letting it pick its preferred format minimises unnecessary conversion on its side.

In [ ]:
def negotiate(
    sender_supported: List[str],
    receiver_supported: List[str],
    sender_pref: List[str],
    receiver_pref: List[str]
) -> Optional[str]:
    s = {x.upper() for x in sender_supported}
    r = {x.upper() for x in receiver_supported}
    overlap = s & r

    if not overlap:
        return None  # no common ground

    # Sender preference takes priority
    for p in [x.upper() for x in sender_pref]:
        if p in overlap:
            return p

    # Fall back to receiver preference
    for p in [x.upper() for x in receiver_pref]:
        if p in overlap:
            return p

    return next(iter(overlap))  # last resort: any common format

print("Negotiation function defined.")

### Verify the negotiation logic independently

Before wiring everything together, it helps to test `negotiate()` in isolation.

In [ ]:
# Case 1: overlap exists, sender prefers XML
result = negotiate(["XML", "JSON"], ["JSON"], ["XML", "JSON"], ["JSON"])
print("Case 1 (sender prefers XML, receiver only JSON):", result)  # expect JSON — XML not in overlap

# Case 2: overlap exists, sender prefers JSON
result = negotiate(["XML", "JSON"], ["XML"], ["JSON", "XML"], ["XML"])
print("Case 2 (sender prefers JSON, receiver only XML):", result)  # expect XML — JSON not in overlap

# Case 3: no overlap
result = negotiate(["XML"], ["JSON"], ["XML"], ["JSON"])
print("Case 3 (no overlap):                           ", result)   # expect None

Notice that in Cases 1 and 2, the sender's *preference* is overridden by the receiver's *constraint* — the agreed format must be something both sides support.

---

## Step 5 — The Agent Class

Each `Agent` has:
- `supported` — the list of formats it can handle
- `pref` — its preference order (first = most preferred)
- `send()` — runs the full negotiate → convert → deliver pipeline
- `receive()` — validates that the arriving format is one it supports

In [ ]:
class Agent:
    def __init__(self, name: str, supported: List[str], pref: List[str]):
        self.name = name
        self.supported = supported
        self.pref = pref

    def encode(self, fmt: str, message: Dict[str, Any]) -> bytes:
        if fmt.upper() == "JSON":
            return dict_to_json_bytes(message)
        elif fmt.upper() == "XML":
            return dict_to_xml_bytes(message)
        raise ValueError(f"Unsupported encode format: {fmt}")

    def receive(self, payload: bytes, fmt: str) -> bool:
        return fmt.upper() in [x.upper() for x in self.supported]

    def send(self, receiver: "Agent", message: Dict[str, Any], src_format: str) -> Dict[str, Any]:
        # --- STEP 2: NEGOTIATE ---
        agreed = negotiate(self.supported, receiver.supported, self.pref, receiver.pref)
        print(f"  NEGOTIATE : sender={self.name}, receiver={receiver.name}")
        print(f"              sender_supported={self.supported}, receiver_supported={receiver.supported}")
        print(f"              agreed={agreed}")

        if not agreed:
            print("  FALLBACK  : No common format. Action=abort_and_log")
            return {"ok": False, "reason": "no_common_format"}

        # --- STEP 3: CONVERT (if needed) ---
        payload_before = self.encode(src_format, message)
        needs_conv = src_format.upper() != agreed.upper()

        if needs_conv:
            print(f"  CONVERT   : {src_format.upper()} -> {agreed.upper()}")
            payload_after = convert_bytes(src_format, agreed, payload_before)
        else:
            print(f"  NO CONVERT: already in agreed format ({agreed.upper()})")
            payload_after = payload_before

        print(f"  PAYLOAD BEFORE ({src_format.upper()}): {payload_before.decode('utf-8')}")
        print(f"  PAYLOAD AFTER  ({agreed.upper()}): {payload_after.decode('utf-8')}")

        # --- STEP 4: DELIVER ---
        content_type = "application/json" if agreed.upper() == "JSON" else "application/xml"
        ok = receiver.receive(payload_after, agreed)
        print(f"  DELIVER   : content_type={content_type}, ok={ok}")

        return {"ok": ok, "content_type": content_type, "agreed_format": agreed}

print("Agent class defined.")

---

## Step 6 — The Test Message

All scenarios use the same payload. This lets you focus on what changes (the format) vs. what stays the same (the data).

In [ ]:
message = {"id": 101, "title": "Quarterly Report", "amount": 123.45}
print("Message:", message)

---

## Scenario A — XML → JSON (from demo3.1)

**Setup:**
- Agent A supports XML and JSON, but **prefers XML** (XML listed first in pref)
- Agent B supports **JSON only**
- A sends in its preferred format: XML

**What to watch:**
- Negotiation finds the only overlap: JSON
- A's preference (XML) is overridden because B can't accept it
- The XML payload is automatically converted to JSON before delivery

In [ ]:
print("=" * 55)
print("SCENARIO A: XML → JSON (sender adapts to receiver)")
print("=" * 55)

A1 = Agent("A", supported=["XML", "JSON"], pref=["XML", "JSON"])  # prefers XML
B1 = Agent("B", supported=["JSON"],        pref=["JSON"])          # JSON only

result_a = A1.send(B1, message, src_format="XML")
print("\nRESULT:", result_a)

**What happened?**

Even though A sent in XML (its preference), the negotiation forced the agreed format to be JSON (the only thing B understands). The conversion happened transparently — B received a correctly-formatted JSON payload and had no idea A originally produced XML.

---

## Scenario B — JSON → XML (from demo3.2)

**Setup:**
- Agent A supports XML and JSON, but **prefers JSON** (JSON listed first in pref)
- Agent B supports **XML only**
- A sends in its preferred format: JSON

**What to watch:**
- Negotiation finds the only overlap: XML
- A's preference (JSON) is overridden because B can't accept it
- The JSON payload is automatically converted to XML before delivery
- The data is identical to Scenario A — only the direction flipped

In [ ]:
print("=" * 55)
print("SCENARIO B: JSON → XML (sender adapts to receiver)")
print("=" * 55)

A2 = Agent("A", supported=["XML", "JSON"], pref=["JSON", "XML"])  # prefers JSON
B2 = Agent("B", supported=["XML"],         pref=["XML"])           # XML only

result_b = A2.send(B2, message, src_format="JSON")
print("\nRESULT:", result_b)

**What happened?**

The mirror image of Scenario A. The protocol is symmetric — it doesn't care which direction the conversion goes. The same negotiation logic and the same conversion router handle both cases.

---

## Scenario C — No Overlap (Safe Fallback)

**Setup:**
- Agent C supports **XML only**
- Agent D supports **JSON only**
- There is no common format

**What to watch:**
- Negotiation returns `None` immediately
- No conversion is attempted — there's nothing to convert *to*
- The system aborts cleanly with a structured error, rather than crashing or sending corrupt data

In [ ]:
print("=" * 55)
print("SCENARIO C: No overlap → safe fallback")
print("=" * 55)

C = Agent("C", supported=["XML"],  pref=["XML"])
D = Agent("D", supported=["JSON"], pref=["JSON"])

result_c = C.send(D, message, src_format="XML")
print("\nRESULT:", result_c)

**What happened?**

The system never touched the payload. It detected the incompatibility at the negotiation step and returned a structured failure — `ok: False, reason: no_common_format`. This is the correct behaviour: a known, auditable failure is far better than a silent corruption or an unhandled exception.

---

## Bonus Scenario — No Conversion Needed

What if both agents already speak the same format and the sender is already using it? The conversion step should be skipped entirely.

In [ ]:
print("=" * 55)
print("BONUS: Both speak JSON, no conversion needed")
print("=" * 55)

E = Agent("E", supported=["JSON"], pref=["JSON"])
F = Agent("F", supported=["JSON"], pref=["JSON"])

result_bonus = E.send(F, message, src_format="JSON")
print("\nRESULT:", result_bonus)

The `NO CONVERT` line confirms the payload passed through unchanged.

---

## Summary — All Scenarios Side by Side

In [ ]:
print(f"{'Scenario':<12} {'src_format':<12} {'agreed':<10} {'ok':<6} {'content_type'}")
print("-" * 60)

rows = [
    ("A (XML→JSON)",  "XML",  result_a.get("agreed_format", "—"), result_a["ok"], result_a.get("content_type", "—")),
    ("B (JSON→XML)",  "JSON", result_b.get("agreed_format", "—"), result_b["ok"], result_b.get("content_type", "—")),
    ("C (fallback)",  "XML",  "None",                              result_c["ok"], "—"),
    ("Bonus (same)",  "JSON", result_bonus.get("agreed_format", "—"), result_bonus["ok"], result_bonus.get("content_type", "—")),
]

for name, src, agreed, ok, ct in rows:
    print(f"{name:<12} {src:<12} {str(agreed):<10} {str(ok):<6} {ct}")

---

## Key Takeaways

| Concept | What it means |
|---|---|
| **Format declaration** | Agents advertise capabilities up front — no assumptions |
| **Negotiation** | The agreed format is the intersection of both sides, biased toward sender preference |
| **Automatic conversion** | The sender adapts; the receiver never needs to know a conversion happened |
| **Safe fallback** | No overlap → abort immediately with a structured error, never corrupt data |
| **Symmetry** | The protocol works identically for XML→JSON and JSON→XML |
| **No-op path** | If both sides already agree, conversion is skipped entirely |

---

## Exercises

### Beginner
1. Add a third format, `CSV`, to the converters and Agent class. Create a scenario where one agent prefers CSV and another prefers JSON.
2. Change Agent A in Scenario A to prefer JSON over XML. What changes in the output?

### Intermediate
3. Add a `priority` integer field to each Agent. Modify the negotiate function to use **receiver** preference first when the receiver has higher priority than the sender.
4. Log every negotiation result to a list, then print an audit trail at the end showing every format decision made in the session.

### Advanced
5. Replace the hardcoded format list with a capability that each agent **advertises at connection time** using a `handshake()` method. Make `send()` call `handshake()` automatically if the agents haven't met before.
6. Extend the system to handle **three-agent chains**: A → B → C, where B acts as a relay and must negotiate with both A and C independently.